# T07 — Retrieval Strategies

The baseline retriever embeds the query and returns the nearest chunks in embedding space. That works when the query is phrased like the documents. It breaks when queries:

- rely on exact identifiers or codes
- mix meaning with keyword constraints
- match many chunks vaguely but only a few precisely
- concern one known document, but search all of them

The most visible symptom is vocabulary mismatch: the right chunk exists but ranks too low.

| User query | Document text | Issue |
|---|---|---|
| `"FSC-C147829 certificate"` | `"reference: FSC-C147829"` | Embeddings blur exact codes |
| `"PFAS-free"` | `"no intentionally added per- and polyfluoroalkyl substances"` | Acronym vs. full name |

This notebook compares three alternatives to the baseline:

| Strategy | Approach | Extra cost |
|---|---|---|
| Baseline | Semantic search | - |
| BM25 | Keyword search, no embeddings | In-memory index |
| Hybrid | Semantic + BM25 via Reciprocal Rank Fusion | In-memory index |
| Metadata filter | Semantic search within known documents | None |

Query rewriting is covered in T09.

## Setup

Requires `OPENAI_API_KEY` (locally: in `.env.local` at the repo root). Run T01 first: it builds the vector store the later notebooks reuse.

In [ ]:
import sys

from loguru import logger
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerRelevancy, Faithfulness
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings as LangChainOpenAIEmbeddings

from conversational_toolkit.evaluation import EvaluationSample
from conversational_toolkit.evaluation.adapters import evaluate_with_ragas
from ki_kompass_rag.advanced_retrieval import (
    compare_retrieval_strategies,
    print_strategy_comparison,
    rag_answer_from_chunks,
    retrieve_baseline,
    retrieve_with_metadata_filter,
)
from ki_kompass_rag.baseline_rag import (
    RETRIEVER_TOP_K,
    build_llm,
    load_or_build_vector_store,
)
from ki_kompass_rag.evaluation import EVALUATION_QUERIES

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
llm = build_llm("openai")

## BM25

**BM25** ranks chunks by exact token matches, weighted by corpus statistics. No embeddings are involved. It combines:

- **Term frequency saturation**: more occurrences of a term help, with diminishing returns.
- **Length normalisation**: long chunks are not favoured just for containing more words.
- **Inverse document frequency**: rare terms (codes, acronyms, product names) weigh more than common words.

Two parameters tune it: `k1` (saturation, typically 1.2 to 2.0) and `b` (length normalisation, 0 to 1). The index (term and document frequencies) is built once from all chunks in the vector store and kept in memory.

BM25 is vocabulary-literal: "carbon footprint decrease" and "CO₂ reduction" share no token, so it treats them as unrelated. Semantic search handles paraphrases; BM25 handles exact codes.

In [ ]:
QUERY = "FSC-C147829 certificate"
KEYWORDS = ["FSC", "C147829"]

results = await compare_retrieval_strategies(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
)
print_strategy_comparison(
    {name: results[name] for name in ("bm25", "baseline")},
    relevant_keywords=KEYWORDS,
    top_n=RETRIEVER_TOP_K,
)

✓ marks chunks that contain one of the keywords.

## Hybrid Retrieval: Semantic + BM25

Hybrid retrieval runs both retrievers in parallel and merges their ranked lists with **Reciprocal Rank Fusion (RRF)**:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}, \quad k = 60$$

RRF uses ranks, not scores, so cosine similarities and BM25 scores never need to be put on one scale. Chunks found by both retrievers rank highest; chunks found by one still count.

In [ ]:
QUERY = "Are any tape products free of per- and polyfluoroalkyl substances?"
KEYWORDS = ["PFAS", "polyfluoroalkyl"]

results = await compare_retrieval_strategies(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
)
print_strategy_comparison(results, relevant_keywords=KEYWORDS, top_n=RETRIEVER_TOP_K)

## Metadata Filtering

The strategies above search all documents. When the question concerns a known document ("what are the transport emissions in the tesa EPD?"), chunks from other documents only add noise. A ChromaDB `where` filter restricts the search to chunks whose metadata match, before ranking:

```python
{"source_file": {"$eq": "EPD_tape_IPG_hotmelt.pdf"}}                 # one file
{"$or": [{"source_file": {"$eq": "A.pdf"}},
         {"source_file": {"$eq": "B.pdf"}}]}                         # two files
```

This fits when the user has selected a document in the UI or a tool passes a document name.

In [ ]:
records = await vector_store.get_chunks_by_filter()
print("\n".join(sorted({r.metadata["source_file"] for r in records})))

In [ ]:
TARGET_FILE = "EPD_cardboard_grupak_corrugated.pdf"
QUERY = "What are the environmental impacts of this product?"

filtered = await retrieve_with_metadata_filter(
    QUERY,
    embedding_model,
    vector_store,
    filters={"source_file": {"$eq": TARGET_FILE}},
    top_k=RETRIEVER_TOP_K,
)
unfiltered = await retrieve_baseline(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
)
print(f"Filtered to {TARGET_FILE}:\n{filtered}\n\nUnfiltered:\n{unfiltered}")

## Answers per Strategy

`rag_answer_from_chunks()` answers from a given list of chunks with the T01 system prompt, so only the retrieval differs.

In [ ]:
QUERY = "What is the FSC-C147829 certificate?"
KEYWORDS = ["FSC", "C147829"]

results = await compare_retrieval_strategies(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
)
print_strategy_comparison(results, relevant_keywords=KEYWORDS, top_n=RETRIEVER_TOP_K)
for name, result in results.items():
    print(f"\n>>> {name}")
    print(await rag_answer_from_chunks(llm, result.chunks, QUERY))

## Answer Quality per Strategy

For every query of the T04 test set, each strategy retrieves chunks and generates an answer; RAGAS then scores the answers with the two reference-free metrics from T04 (Faithfulness, AnswerRelevancy). This makes 21 answers plus the judge calls, a few minutes.

In [ ]:
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", max_completion_tokens=8192)
)
ragas_embeddings = LangChainOpenAIEmbeddings(model="text-embedding-3-small")

STRATEGIES = ("baseline", "bm25", "hybrid")
samples: dict[str, list[EvaluationSample]] = {s: [] for s in STRATEGIES}
for query in (q["query"] for q in EVALUATION_QUERIES):
    results = await compare_retrieval_strategies(
        query, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
    )
    for strategy in STRATEGIES:
        chunks = list(results[strategy].chunks)
        samples[strategy].append(
            EvaluationSample(
                query=query,
                answer=await rag_answer_from_chunks(llm, chunks, query),
                retrieved_chunks=chunks,
            )
        )

In [ ]:
metrics = [Faithfulness(), AnswerRelevancy(strictness=1)]
print(f"{'strategy':<10}" + "".join(f"{m.name:>20}" for m in metrics))
for strategy, strategy_samples in samples.items():
    report = evaluate_with_ragas(
        samples=strategy_samples,
        metrics=metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    print(f"{strategy:<10}" + "".join(f"{s:>20.3f}" for s in report.summary().values()))

## Summary

| Strategy | Best for | Limitation |
|---|---|---|
| Baseline (semantic) | Paraphrases, vocabulary mismatch | Exact codes and IDs |
| BM25 | Exact terms, product codes, acronyms | Paraphrases, synonyms |
| Hybrid | Both, consistently | Index must fit in memory |
| Metadata filter | Questions about a known document | Caller must know the document |